# PA 766 · Module 8 · Prompting-based text classification

**Question:** How does the classification approach affect sentiment predictions?

Compare four configurations on the same 300 tweets: **RoBERTa sentiment**, **FLAN-T5 zero-shot**, **FLAN-T5 one-shot**, and **FLAN-T5 three-shot**.

The FLAN-T5 configurations use the same model with zero, one, or three labeled examples. All model weights stay fixed.

**Workflow:** inspect predictions → compare prompts → classify tweets → compare metrics → read disagreements.

## 1 · Setup and data

In **Google Colab**, select **Runtime → Change runtime type → T4 GPU** if available. Open a copy and run cells in order. Models download on first use; CPU inference is slower.

Install once in a fresh session. If Colab requests a restart, restart and continue with the imports.


In [ ]:
%pip install -q "transformers==4.57.6" "torch>=2.6,<3" "pandas>=2.2,<3" "scikit-learn>=1.5,<2" "matplotlib>=3.9,<4" sentencepiece
# Installs the packages used in this notebook.

In [ ]:
from pathlib import Path  # Handles file paths.

import matplotlib.pyplot as plt  # Creates plots.
import pandas as pd  # Works with tables.
import torch  # Runs the models on CPU or GPU.
from IPython.display import display  # Displays tables in the notebook.
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay  # Calculates metrics and confusion matrices.
from transformers import pipeline  # Loads pretrained models.

In [ ]:
LABELS = ["negative", "neutral", "positive"]  # Sets the three sentiment categories.
BATCH_SIZE = 8  # Processes eight tweets at a time.
if torch.cuda.is_available():  # Checks for a CUDA GPU.
    DEVICE = 0  # Uses the first CUDA GPU.
elif torch.backends.mps.is_available():  # Checks for an Apple GPU.
    DEVICE = "mps"  # Uses the Apple GPU locally.
else:
    DEVICE = -1  # Uses the CPU.
pd.set_option("display.max_colwidth", 180)  # Shows longer text in tables.
print("Device:", DEVICE, "(0 = Colab GPU; mps = Apple GPU; -1 = CPU)")  # Displays the selected device.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
DATA_DIR = Path("/content/drive/MyDrive/a-ncsu-courses/PA766-2026/Data/Tweet Eval Sentiment")

development = pd.read_csv(DATA_DIR / "tweeteval_sentiment_development_60.csv")
evaluation = pd.read_csv(DATA_DIR / "tweeteval_sentiment_evaluation_300.csv")

In [ ]:
for table in [development, evaluation]:
    # Replace usernames and URLs using the model card's preprocessing.
    table["model_text"] = table["text"].str.replace(r"(?<!\S)@\S+", "@user", regex=True)
    table["model_text"] = table["model_text"].str.replace(r"(?<!\S)http\S*", "http", regex=True)

results = evaluation.copy()
print("Loaded:", len(development), "development tweets and", len(evaluation), "evaluation tweets")

### What are we measuring?

**TweetEval sentiment** contains annotated tweets on varied topics. We classify **overall sentiment**, which can differ from support for a policy. A tweet can sound positive while opposing a policy.

| File | Contents | Role |
|---|---|---|
| `tweeteval_sentiment_development_60.csv` | 60 validation tweets; 20 per class | Preview the task and supply prompt examples |
| `tweeteval_sentiment_evaluation_300.csv` | 300 test tweets | Compare all four configurations |

Reference labels are human annotations. Context, sarcasm, and mixed sentiment can make them debatable. Preparation checked ID and normalized-text overlap between sets; prior model exposure remains unknown.

All four configurations receive the same processed text; the original text stays available for interpretation.

The 12-row illustrative metrics file contains invented examples and is not used here.

In [ ]:
class_counts = pd.DataFrame({  # Creates a class-count table.
    "development": development["label"].value_counts(),  # Counts development labels.
    "evaluation": evaluation["label"].value_counts(),  # Counts evaluation labels.
}).reindex(LABELS)  # Orders rows by sentiment.
display(class_counts)  # Displays the counts.

In [ ]:
practice = pd.concat([  # Combines five practice tweets.
    development[development["label"] == label].head(number)  # Selects tweets for each class.
    for label, number in [("negative", 2), ("neutral", 1), ("positive", 2)]  # Sets the number per class.
], ignore_index=True)  # Resets the row index.

practice

## 2 · Use a task-specific RoBERTa classifier

`cardiffnlp/twitter-roberta-base-sentiment` was fine-tuned for three sentiment categories. Its classification head has fixed labels. We load it and run inference.

The learned decision rule maps each tweet to one of the three categories.

**Text → scores → highest-scoring label.**

We map `LABEL_0`, `LABEL_1`, and `LABEL_2` to sentiment names. A high score does not guarantee correctness.


In [ ]:
ROBERTA_ID = "cardiffnlp/twitter-roberta-base-sentiment"  # Selects the sentiment model.
ROBERTA_REVISION = "daefdd1f6ae931839bce4d0f3db0a1a4265cd50f"  # Fixes the model version.
ROBERTA_LABEL_MAP = {"LABEL_0": "negative", "LABEL_1": "neutral", "LABEL_2": "positive"}  # Maps model IDs to sentiments.

roberta = pipeline(  # Creates the sentiment classifier.
    "text-classification", model=ROBERTA_ID, revision=ROBERTA_REVISION, device=DEVICE,  # Loads the fixed model on the selected device.
)
print("Model label mapping:", roberta.model.config.id2label)  # Displays the model's labels.

In [ ]:
example_text = practice.loc[0, "model_text"]  # Selects the first practice tweet.
print("Tweet:", example_text)  # Displays the model input.

In [ ]:
one_roberta = roberta(example_text, top_k=None, truncation=True, max_length=512)  # Returns all label scores, limiting input to 512 tokens.

In [ ]:
display(pd.DataFrame([  # Displays the scores as a table.
    {"label": ROBERTA_LABEL_MAP[item["label"]], "score": item["score"]}  # Uses sentiment names and scores.
    for item in one_roberta  # Reads each label's result.
]))

### Classify all evaluation tweets

In [ ]:
raw_roberta = roberta(  # Classifies the evaluation tweets.
    results["model_text"].tolist(), top_k=None, batch_size=BATCH_SIZE,  # Supplies tweets and returns all label scores in batches.
    truncation=True, max_length=512,  # Limits each input to 512 tokens.
)

In [ ]:
roberta_scores = pd.DataFrame([  # Creates one score row per tweet.
    {ROBERTA_LABEL_MAP[item["label"]]: item["score"] for item in row}  # Maps label IDs to sentiment-score columns.
    for row in raw_roberta  # Reads each tweet's results.
])[LABELS]  # Orders the sentiment columns.

In [ ]:
results["roberta_prediction"] = roberta_scores.idxmax(axis=1)  # Selects the highest-scoring label.
results["roberta_score"] = roberta_scores.max(axis=1)  # Saves its score.
display(results[["text", "label", "roberta_prediction"]].head())  # Displays the first five predictions.

## 3 · Use FLAN-T5 for zero-shot classification

FLAN-T5-base is an instruction-tuned encoder–decoder model. It generates an answer from a text prompt; its weights stay fixed.

The prompt gives instructions, category definitions, and the tweet to classify. No labeled example tweets are supplied.

Use answer codes **A = negative**, **B = neutral**, and **C = positive**. Generate one token, map the answer to a sentiment label, and retain unexpected answers as `invalid`.

In [ ]:
MODEL_ID = "google/flan-t5-base"
MODEL_REVISION = "c782cba52f8ea6a704240578055cf1c3fc2f2ca9"  # Same version as the DSA495 notebook.
classifier = pipeline(
    "text2text-generation", model=MODEL_ID,
    revision=MODEL_REVISION, device=DEVICE,
)

CODE_TO_LABEL = {"A": "negative", "B": "neutral", "C": "positive"}
LABEL_TO_CODE = {label: code for code, label in CODE_TO_LABEL.items()}

INSTRUCTIONS = (
    "Classify the overall sentiment of the tweet. Answer with one letter: A, B, or C.\n"
    "A: Negative sentiment, expressing criticism, dissatisfaction, or unhappiness.\n"
    "B: Neutral sentiment, giving information without clear positive or negative emotion.\n"
    "C: Positive sentiment, expressing praise, satisfaction, or happiness.\n\n"
)

def make_zero_prompt(text):
    return INSTRUCTIONS + f"Tweet: {text}\nAnswer:"

def classify_prompts(prompts):
    outputs = classifier(
        prompts, max_new_tokens=1, do_sample=False, batch_size=BATCH_SIZE,
    )
    return [row["generated_text"] for row in outputs]

def read_label(answer):
    return CODE_TO_LABEL.get(answer.strip().upper(), "invalid")

In [ ]:
print(make_zero_prompt(example_text))  # Shows the complete zero-shot prompt.

In [ ]:
one_flan = classify_prompts([make_zero_prompt(example_text)])[0]
print("Raw answer:", one_flan)
print("Predicted sentiment:", read_label(one_flan))

**Check:** Why is this zero-shot despite the category definitions? How does its input differ from RoBERTa's?

Category definitions describe possible answers; they are not labeled example tweets. `do_sample=False` uses greedy decoding. The same model, instructions, answer codes, and generation settings will be used for zero-shot, one-shot, and three-shot classification.

### Optional · Explore wording before evaluation

On development tweets, edit a category definition and inspect changed predictions. This changes the input, not the weights.

For the required comparison, keep `INSTRUCTIONS` and `CODE_TO_LABEL` fixed. Choose wording before evaluating held-out tweets.

In [ ]:
zero_prompts = [make_zero_prompt(text) for text in results["model_text"]]
results["zero_shot_answer"] = classify_prompts(zero_prompts)
results["zero_shot_prediction"] = results["zero_shot_answer"].map(read_label)
print("Zero-shot invalid answers:", (results["zero_shot_prediction"] == "invalid").sum())
display(results[["text", "label", "roberta_prediction", "zero_shot_answer", "zero_shot_prediction"]].head())

## 4 · Compare one-shot and three-shot prompts

Add labeled tweet–answer demonstrations to the same FLAN-T5 prompt.

- **One-shot:** one labeled neutral tweet (one demonstration total).
- **Three-shot:** the same neutral tweet plus one negative and one positive tweet (three demonstrations total).

Examples come only from the development set. The instructions, model, and generation settings stay fixed; only the demonstrations change. Freeze these prompts before evaluation. No model training occurs.

In [ ]:
three_shot_examples = pd.concat([
    development[development["label"] == label].head(1)
    for label in ["neutral", "negative", "positive"]
], ignore_index=True)
one_shot_examples = three_shot_examples.head(1).copy()
display(three_shot_examples[["example_id", "text", "label"]])

# Preview a development tweet that is not a demonstration.
prompt_example_text = development.loc[
    ~development["example_id"].isin(three_shot_examples["example_id"]), "model_text"
].iloc[0]

In [ ]:
def build_demonstrations(rows):  # Build labeled examples for the prompt
    demonstrations = ""  # Start with an empty string
    for row in rows.itertuples():  # Loop through the example rows
        demonstrations += f"Tweet: {row.model_text}\nAnswer: {LABEL_TO_CODE[row.label]}\n\n"  # Add each tweet and its label code
    return demonstrations  # Return the formatted examples

one_shot_demonstrations = build_demonstrations(one_shot_examples)  # Format the one-shot examples
three_shot_demonstrations = build_demonstrations(three_shot_examples)  # Format the three-shot example

In [ ]:
def make_few_prompt(text, demonstrations):  # Combine instructions, examples, and a new tweet
    return INSTRUCTIONS + demonstrations + f"Tweet: {text}\nAnswer:"  # Leave the final answer for the model

one_shot_prompt = make_few_prompt(prompt_example_text, one_shot_demonstrations)  # Create the one-shot prompt
three_shot_prompt = make_few_prompt(prompt_example_text, three_shot_demonstrations)  # Create the three-shot prompt

In [ ]:
# Show all three prompts for the same target tweet.
for name, prompt in [
    ("ZERO-SHOT", make_zero_prompt(prompt_example_text)),
    ("ONE-SHOT", one_shot_prompt),
    ("THREE-SHOT", three_shot_prompt),
]:
    print("\n" + name)
    print(prompt)

In [ ]:
preview_rows = []
for name, prompt in [
    ("FLAN-T5 zero-shot", make_zero_prompt(prompt_example_text)),
    ("FLAN-T5 one-shot", one_shot_prompt),
    ("FLAN-T5 three-shot", three_shot_prompt),
]:
    answer = classify_prompts([prompt])[0]
    preview_rows.append({"configuration": name, "answer": answer, "prediction": read_label(answer)})
print("Tweet:", prompt_example_text)
display(pd.DataFrame(preview_rows))

**Discuss:** What information does the three-shot prompt add? Does adding examples always improve predictions?

### Classify all evaluation tweets

Use the same fixed demonstrations for every evaluation tweet. Evaluation labels never enter a prompt. Unexpected answers count as errors; they are not dropped.

In [ ]:
for prefix, demonstrations in [  # Run classification with each set of examples
    ("one_shot", one_shot_demonstrations),  # Use the one-shot examples
    ("three_shot", three_shot_demonstrations),  # Use the three-shot examples
]:
    prompts = [make_few_prompt(text, demonstrations) for text in results["model_text"]]  # Build a prompt for each tweet
    results[prefix + "_answer"] = classify_prompts(prompts)  # Save the model's raw answers
    results[prefix + "_prediction"] = results[prefix + "_answer"].map(read_label)  # Convert answers into labels
    print(prefix, "invalid answers:", (results[prefix + "_prediction"] == "invalid").sum())  # Count unrecognized answers

In [ ]:
display(results[[  # Show tweets, true labels, and model predictions
    "text", "label", "roberta_prediction", "zero_shot_prediction",  # Include baseline predictions
    "one_shot_prediction", "three_shot_prediction",  # Include few-shot predictions
]].head())  # Display the first five rows

## 5 · Compare all four configurations

- **Accuracy:** fraction matching reference labels.
- **Precision:** among tweets predicted as a class, the fraction with that reference label.
- **Recall:** among reference-class tweets, the fraction identified correctly.
- **F1:** combines precision and recall.
- **Macro-F1:** averages class F1 scores equally, limiting the largest class's influence.

The evaluation set has **96 negative, 141 neutral, and 63 positive tweets**. Read class results alongside overall scores.

Neutral is the largest class, so accuracy alone can conceal weaknesses in other classes.

Model sizes and training histories differ. The three FLAN-T5 configurations share model weights, instructions, answer codes, and decoding settings, allowing a comparison of their prompts. This experiment changes both example count and class coverage; it does not establish that more examples always help.

In [ ]:
METHODS = {  # Match each method to its prediction column
    "RoBERTa sentiment": "roberta_prediction",  # RoBERTa predictions
    "FLAN-T5 zero-shot": "zero_shot_prediction",  # Predictions without examples
    "FLAN-T5 one-shot": "one_shot_prediction",  # Predictions with one-shot examples
    "FLAN-T5 three-shot": "three_shot_prediction",  # Predictions with three-shot examples
}
comparison_rows = []  # Store evaluation results for each method
for method, column in METHODS.items():  # Evaluate each prediction column
    comparison_rows.append({  # Add this method's evaluation results
        "configuration": method,  # Record the method name
        "accuracy": accuracy_score(results["label"], results[column]),  # Calculate the fraction of correct predictions
        "macro_F1": f1_score(results["label"], results[column], labels=LABELS, average="macro", zero_division=0),  # Average F1 equally across labels
        "invalid_answers": (results[column] == "invalid").sum(),  # Count unrecognized answers
    })
comparison = pd.DataFrame(comparison_rows).set_index("configuration")  # Create a table indexed by method
display(comparison.round(3))  # Display scores rounded to three decimal places

In [ ]:
for method, column in METHODS.items():  # Creates a report for each model.
    report = classification_report(  # Calculates class-level metrics.
        results["label"], results[column], labels=LABELS, output_dict=True, zero_division=0,  # Compares reference labels and predictions.
    )
    print(method)  # Displays the model name.
    display(pd.DataFrame(report).T.loc[LABELS, ["precision", "recall", "f1-score", "support"]].round(3))  # Displays metrics and reference-class counts.


In [ ]:
matrix_labels = LABELS.copy()
if results[list(METHODS.values())].eq("invalid").any().any():
    matrix_labels.append("invalid")  # Shows unexpected generated answers in the matrices.

fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for (method, column), ax in zip(METHODS.items(), axes.flat):
    matrix = ConfusionMatrixDisplay.from_predictions(
        results["label"], results[column], labels=matrix_labels,
        cmap="Blues", colorbar=False, values_format="d", ax=ax,
    )
    matrix.im_.set_clim(0, results["label"].value_counts().max())
    ax.set_title(method)
    ax.set_xlabel("Predicted sentiment")
    ax.set_ylabel("Reference sentiment")
fig.suptitle("Same 300 evaluation tweets · counts")
fig.tight_layout()
plt.show()

**Discuss**

1. Which configuration has the highest accuracy? Highest macro-F1?
2. Does FLAN-T5 three-shot outperform FLAN-T5 one-shot? What changed in the prompt?
3. Which class has the lowest recall for each model? Find its mistakes in the matrix.
4. When identifying negative public-service comments, what would false negatives and false positives mean?

Rows are reference labels; columns are predictions. For a class, the diagonal counts correct predictions, the rest of its row counts false negatives, and the rest of its column counts false positives.

If an `invalid` category appears, it represents an unexpected model answer; there are no reference tweets in that category.

## 6 · Read disagreements

The table shows differing predictions. These may reflect errors, ambiguous annotations, or different interpretations. Metrics use **all 300 tweets**, including shared correct predictions and mistakes.

In [ ]:
prediction_columns = list(METHODS.values())
disagreements = results[results[prediction_columns].nunique(axis=1) > 1]
print("Tweets with different predictions:", len(disagreements), "of", len(results))
with pd.option_context("display.max_colwidth", None):
    display(disagreements[["example_id", "text", "label"] + prediction_columns].head(5))

### Sources

- [TweetEval: dataset and benchmark](https://github.com/cardiffnlp/tweeteval) · [paper](https://aclanthology.org/2020.findings-emnlp.148/). The classroom CSVs retain source-split names, row indices, and example IDs. TweetEval is distributed under CC BY 3.0; the source repository also notes Twitter's terms.
- [RoBERTa sentiment model card](https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment): sentiment mapping and username/URL preprocessing.
- [FLAN-T5-base model card](https://huggingface.co/google/flan-t5-base): instruction-tuned model used for zero-shot, one-shot, and three-shot classification.
- [FLAN instruction-tuning paper](https://arxiv.org/abs/2210.11416).